In [ ]:
import json 
import pandas as pd
import numpy 
import matplotlib.pyplot as plt 
import seaborn as sns

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
# input_path = 'arxiv_sample_50k.json'
# data = []
# with open(input_path, 'r') as f:
#     for line in f:
#         data.append(json.loads(line))
df = pd.read_json('arxiv_sample_50k.json', lines = True)


In [3]:
df.head()

,id,authors,title,categories,abstract
0,2303.06576,"E. Grohs, A. B. Balantekin",Implications on Cosmology from Dirac Neutrino ...,hep-ph astro-ph.CO,The mechanism for generating neutrino masses...
1,1101.3145,"H.Q. Yuan, J. Chen, J. Singleton, S. Akutagawa...",Large upper critical field in non-centrosymmet...,cond-mat.supr-con,We determine the upper critical field $\mu_0...
2,1506.05616,"Jaya Kumar A., Buddhapriya Chakrabarti and Yas...",Equilibrium of fluid membranes with tangent-pl...,cond-mat.soft,Fluid membranes endowed with tangent-plane o...
3,2408.06921,"Chao Yu, Qi Xu, and Jun Zhang",Recent advances in InGaAs/InP single-photon de...,physics.ins-det quant-ph,Single-photon detectors (SPDs) are widely us...
4,2508.17313,"Alexander Alban, Fu-Hsuan Ho, Justin Ko",The Free Energy of an Enriched Continuous Rand...,math.PR math.AP,We revisit the proof of the limiting free ener...


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          50000 non-null  object
 1   authors     50000 non-null  object
 2   title       50000 non-null  object
 3   categories  50000 non-null  object
 4   abstract    50000 non-null  object
dtypes: object(5)
memory usage: 1.9+ MB
None


In [5]:
print(sum(df.duplicated()))

0


In [6]:
df['paper_info'] = df['title'] + ' ' + df['abstract']

In [7]:
columns = ['title' , 'abstract']
df = df.drop(columns, axis = 1)


In [8]:
df.head()

,id,authors,categories,paper_info
0,2303.06576,"E. Grohs, A. B. Balantekin",hep-ph astro-ph.CO,Implications on Cosmology from Dirac Neutrino ...
1,1101.3145,"H.Q. Yuan, J. Chen, J. Singleton, S. Akutagawa...",cond-mat.supr-con,Large upper critical field in non-centrosymmet...
2,1506.05616,"Jaya Kumar A., Buddhapriya Chakrabarti and Yas...",cond-mat.soft,Equilibrium of fluid membranes with tangent-pl...
3,2408.06921,"Chao Yu, Qi Xu, and Jun Zhang",physics.ins-det quant-ph,Recent advances in InGaAs/InP single-photon de...
4,2508.17313,"Alexander Alban, Fu-Hsuan Ho, Justin Ko",math.PR math.AP,The Free Energy of an Enriched Continuous Rand...


In [9]:
len(df['categories'].unique())

7007

In [10]:
df['info'] = df['paper_info'] + ' ' + 'Categories' + ' ' + df['categories']

In [11]:
df = df.drop(columns=['paper_info', 'authors', 'categories'])

In [12]:
df.head()

,id,info
0,2303.06576,Implications on Cosmology from Dirac Neutrino ...
1,1101.3145,Large upper critical field in non-centrosymmet...
2,1506.05616,Equilibrium of fluid membranes with tangent-pl...
3,2408.06921,Recent advances in InGaAs/InP single-photon de...
4,2508.17313,The Free Energy of an Enriched Continuous Rand...


In [13]:
df['info'] = df['info'].str.replace('\n', ' ')

df['info'] = df['info'].str.strip()

df['info'] = df['info'].str.lower()

In [14]:
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Load the Tokenizer and the Model
model_name = "sentence-transformers/all-MiniLM-L6-v2"

print("Downloading/Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

Downloading/Loading tokenizer and model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
import torch
from tqdm.auto import tqdm # Import tqdm to create the progress bar

texts = df['info'].tolist()
all_embeddings = []
batch_size = 16 # Process 16 sentences at a time to save memory

print(f"Total texts to process: {len(texts)}")

# 1. Wrap your range() inside tqdm() and give it a description
for i in tqdm(range(0, len(texts), batch_size), desc="Extracting Embeddings"):
    batch_texts = texts[i:i + batch_size]
    
    # 2. Tokenize just the current batch
    encoded_input = tokenizer(batch_texts, padding=True, truncation=True, return_tensors='pt')
    
    # 3. Pass the batch through the model
    with torch.no_grad():
        model_output = model(**encoded_input)
        
    # 4. Apply mean pooling to the batch
    token_embeddings = model_output[0] 
    input_mask_expanded = encoded_input['attention_mask'].unsqueeze(-1).expand(token_embeddings.size()).float()
    batch_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    # 5. Add the batch embeddings to our master list
    all_embeddings.extend(batch_embeddings.tolist())

# 6. Add the final list of embeddings back to the DataFrame
df['embeddings'] = all_embeddings

print("Success! Here is a look at your updated DataFrame:")
print(df.head())

Total texts to process: 50000


Extracting Embeddings:   0%|          | 0/3125 [00:00<?, ?it/s]

Success! Here is a look at your updated DataFrame:
           id                                               info  \
0  2303.06576  implications on cosmology from dirac neutrino ...   
1   1101.3145  large upper critical field in non-centrosymmet...   
2  1506.05616  equilibrium of fluid membranes with tangent-pl...   
3  2408.06921  recent advances in ingaas/inp single-photon de...   
4  2508.17313  the free energy of an enriched continuous rand...   

                                          embeddings  
0  [-0.21436235308647156, -0.12343166768550873, 0...  
1  [-0.230428084731102, -0.151688814163208, -0.17...  
2  [-0.10004530847072601, -0.2825489938259125, -0...  
3  [-0.21548885107040405, 0.11537788063287735, 0....  
4  [-0.15607544779777527, -0.011899984441697598, ...  


In [16]:
df.head()

,id,info,embeddings
0,2303.06576,implications on cosmology from dirac neutrino ...,"[-0.21436235308647156, -0.12343166768550873, 0..."
1,1101.3145,large upper critical field in non-centrosymmet...,"[-0.230428084731102, -0.151688814163208, -0.17..."
2,1506.05616,equilibrium of fluid membranes with tangent-pl...,"[-0.10004530847072601, -0.2825489938259125, -0..."
3,2408.06921,recent advances in ingaas/inp single-photon de...,"[-0.21548885107040405, 0.11537788063287735, 0...."
4,2508.17313,the free energy of an enriched continuous rand...,"[-0.15607544779777527, -0.011899984441697598, ..."


In [17]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Convert the list of embeddings into a 2D numpy array (50000 rows, 384 columns)
embedding_matrix = np.array(df['embeddings'].tolist())

In [20]:
def get_recommendations(target_id, df, embedding_matrix, top_n=5):
    try:
        idx = df[df['id'] == target_id].index[0]
    except IndexError:
        return "Paper ID not found."

    target_embedding = embedding_matrix[idx].reshape(1, -1)
    similarity_scores = cosine_similarity(target_embedding, embedding_matrix)[0]
    top_indices = similarity_scores.argsort()[::-1][1:top_n+1]

    # Changed this line to use 'id' and 'info' since 'title' and 'categories' are dropped
    recommended_papers = df.iloc[top_indices][['id', 'info']].copy()
    
    recommended_papers['similarity_score'] = similarity_scores[top_indices]

    return recommended_papers

# --- Testing the code ---
sample_id = df['id'].iloc[0] 

# Slicing [:150] so it only prints the first 150 characters of the 'info' block
target_info = df[df['id'] == sample_id]['info'].values[0]
print(f"Target Paper Info: {target_info[:150]}...\n")

print("Recommendations:")
recommendations = get_recommendations(sample_id, df, embedding_matrix, top_n=5)
print(recommendations)

Target Paper Info: implications on cosmology from dirac neutrino magnetic moments   the mechanism for generating neutrino masses remains a puzzle in particle physics. if...

Recommendations:
               id                                               info  \
44970   0801.2643  probing neutrino magnetic moment and unparticl...   
23346   1312.2288  creation of dirac neutrinos in a dense medium ...   
6000   2102.07991  manifestations of non-zero majorana cp violati...   
12263   0906.1611  pseudo-dirac neutrinos in the new standard mod...   
16658  1705.04419  high-energy neutrinos from multi-body decaying...   

       similarity_score  
44970          0.728161  
23346          0.683076  
6000           0.676309  
12263          0.664507  
16658          0.660155  


In [21]:
import numpy as np

# 1. Save the 2D numpy matrix
np.save('arxiv_embedding_matrix.npy', embedding_matrix)

# 2. Save the pandas DataFrame
# We use 'pickle' instead of CSV because it perfectly preserves complex data types
df.to_pickle('arxiv_dataframe.pkl')

print("Success! Your files are saved.")

Success! Your files are saved.
